# Cluster ↔ Video Performance Correlation

This notebook measures how strongly within-channel semantic clusters (from category quantization output) are associated with video performance. It computes a documented metric set per channel, and produces an ordered ranking from **most predictive** to **least predictive** using a primary metric.


## 1) Setup and inputs

This cell imports analysis dependencies and defines paths for the category quantization JSON export and optional CSV outputs.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

DATA_PATH = Path("../../business_cluster_video_embeddings_clustered_2d.json")
OUTPUT_DIR = Path("../../exports/cluster_performance")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_ORDERING_METRIC = "adj_r2"  # Used to rank channels from most to less predictive.
MIN_VIDEOS_PER_CHANNEL = 10
MIN_CLUSTERS_PER_CHANNEL = 2


## 2) Load and validate exported cluster JSON

This cell loads the category-quantization JSON and validates required columns so the downstream metrics are reproducible and explicit.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected JSON export at: {DATA_PATH.resolve()}")

df = pd.read_json(DATA_PATH)
required_cols = {"channel_name", "cluster_id", "cluster_name", "view_count", "video_title", "video_url"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Ensure numeric performance target and remove missing rows that block modeling.
df["view_count"] = pd.to_numeric(df["view_count"], errors="coerce")
df = df.dropna(subset=["channel_name", "cluster_id", "view_count"]).copy()
df["cluster_id"] = df["cluster_id"].astype(str)

# Log-transform to stabilize heavy-tailed view distributions.
df["log_views"] = np.log1p(df["view_count"])

print(f"Rows loaded: {len(df):,}")
print(f"Channels: {df['channel_name'].nunique():,}")
print(df.head(3))


## 3) Metric definitions

This cell defines one function that computes a full channel-level metric bundle:

- **ANOVA p-value**: whether mean log-views differ across clusters.
- **R² / Adjusted R²** from `log_views ~ C(cluster_id)`.
- **Eta-squared (η²)**: ANOVA effect size = between-cluster sum of squares / total sum of squares.
- **Kruskal-Wallis p-value**: non-parametric alternative to ANOVA.
- **Epsilon-squared (ε²)** for Kruskal: non-parametric effect size.
- **Cluster median spread ratio**: max cluster median / min cluster median (on raw views).

The notebook uses **Adjusted R² (`adj_r2`)** as the primary ordering metric.


In [ ]:
def channel_metrics(channel_df: pd.DataFrame) -> dict:
    channel_name = channel_df["channel_name"].iloc[0]
    n_videos = len(channel_df)
    n_clusters = channel_df["cluster_id"].nunique()

    # Guardrails for statistical validity.
    if n_videos < MIN_VIDEOS_PER_CHANNEL or n_clusters < MIN_CLUSTERS_PER_CHANNEL:
        return {
            "channel_name": channel_name,
            "n_videos": n_videos,
            "n_clusters": n_clusters,
            "anova_pvalue": np.nan,
            "r2": np.nan,
            "adj_r2": np.nan,
            "eta_squared": np.nan,
            "kruskal_pvalue": np.nan,
            "epsilon_squared": np.nan,
            "median_spread_ratio": np.nan,
            "eligible": False,
            "note": "Insufficient videos or clusters"
        }

    model = smf.ols("log_views ~ C(cluster_id)", data=channel_df).fit()
    anova_tbl = sm.stats.anova_lm(model, typ=2)

    ss_between = float(anova_tbl.loc["C(cluster_id)", "sum_sq"])
    ss_total = ss_between + float(anova_tbl.loc["Residual", "sum_sq"])
    eta_squared = (ss_between / ss_total) if ss_total > 0 else np.nan
    anova_p = float(anova_tbl.loc["C(cluster_id)", "PR(>F)"])

    groups_log = [g["log_views"].values for _, g in channel_df.groupby("cluster_id")]
    kw_stat, kw_p = stats.kruskal(*groups_log)

    k = n_clusters
    n = n_videos
    epsilon_sq = (kw_stat - k + 1) / (n - k) if (n - k) > 0 else np.nan

    cluster_medians = channel_df.groupby("cluster_id")["view_count"].median()
    min_med = cluster_medians.min()
    max_med = cluster_medians.max()
    spread_ratio = (max_med / min_med) if min_med > 0 else np.nan

    return {
        "channel_name": channel_name,
        "n_videos": n_videos,
        "n_clusters": n_clusters,
        "anova_pvalue": anova_p,
        "r2": float(model.rsquared),
        "adj_r2": float(model.rsquared_adj),
        "eta_squared": eta_squared,
        "kruskal_pvalue": float(kw_p),
        "epsilon_squared": float(epsilon_sq),
        "median_spread_ratio": float(spread_ratio) if pd.notna(spread_ratio) else np.nan,
        "eligible": True,
        "note": "ok"
    }


## 4) Compute metrics for each channel

This cell runs the metric function across all channels and creates a complete channel-level metrics table.


In [ ]:
metrics_df = pd.DataFrame([
    channel_metrics(g)
    for _, g in df.groupby("channel_name", sort=True)
])

metrics_df = metrics_df.sort_values(["eligible", PRIMARY_ORDERING_METRIC], ascending=[False, False])
metrics_df.reset_index(drop=True, inplace=True)

metrics_df.head(10)


## 5) Ordered list of channels (most predictive → less predictive)

This cell displays the ranked channel list using **Adjusted R²** as the primary metric, while keeping the full metric set visible.


In [ ]:
ranked = metrics_df[metrics_df["eligible"]].copy()
ranked = ranked.sort_values(PRIMARY_ORDERING_METRIC, ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

display_cols = [
    "rank", "channel_name", "n_videos", "n_clusters",
    "adj_r2", "r2", "eta_squared", "anova_pvalue",
    "epsilon_squared", "kruskal_pvalue", "median_spread_ratio"
]

ranked[display_cols]


## 6) Summary diagnostics

This cell reports aggregate diagnostics to interpret whether clusters are broadly predictive across channels.


In [ ]:
eligible = metrics_df[metrics_df["eligible"]].copy()
if len(eligible) == 0:
    print("No channels passed eligibility thresholds.")
else:
    share_significant = (eligible["anova_pvalue"] < 0.05).mean()
    print(f"Eligible channels: {len(eligible)}")
    print(f"Median adj R²: {eligible['adj_r2'].median():.4f}")
    print(f"Mean adj R²:   {eligible['adj_r2'].mean():.4f}")
    print(f"ANOVA significant (p<0.05): {share_significant:.1%}")


## 7) Optional exports (CSV)

This cell saves the complete metric table and ranked view to CSV so results can be shared downstream.


In [ ]:
all_metrics_csv = OUTPUT_DIR / "channel_cluster_performance_metrics.csv"
ranked_csv = OUTPUT_DIR / "channel_cluster_performance_ranked_by_adj_r2.csv"

metrics_df.to_csv(all_metrics_csv, index=False)
ranked.to_csv(ranked_csv, index=False)

print(f"Wrote: {all_metrics_csv.resolve()}")
print(f"Wrote: {ranked_csv.resolve()}")
